# Post-Project Review — PodcastIQ
**Lab: Stakeholders & Real-World Scope**  
*Ironhack — Week 2, April 2026*

---

## Part 1 - Project Snapshot

**Problem:** Creating a podcast episode from a written article, research PDF, or YouTube talk normally takes hours of manual editing, voice recording, and audio production, which most people simply skip.

**Intended user:** Content creators, educators, and AI researchers who want to repurpose existing written or video content into an audio-first format without any production overhead.

**What we built:** PodcastIQ is a Gradio web application that accepts four source types (plain text, article URL, YouTube link, PDF) and returns a fully produced two-host podcast episode with MP3 audio, full transcript, and episode metadata, in under 60 seconds. The pipeline runs Claude Sonnet or GPT-4o for script generation, a custom humanizer layer for natural speech patterns, and Microsoft edge-tts (with optional ElevenLabs upgrade) for voice synthesis. A content safety guard blocks racist, hateful, or abusive inputs before they reach the LLM, and every run is logged to a JSONL file with timing, settings, and the generated script.

---
## Part 2  Stakeholder Impact

A stakeholder is anyone who can **affect** or be **affected by** the project.

| # | Stakeholder | Role / Relationship | What they need from the outcome | Risk if ignored | Influence | Interest |
|---|---|---|---|---|---|---|
| 1 | **Content Creator / End User** | Direct daily user of the Gradio UI | A reliable, fast tool that produces publish-quality audio from their content | They abandon the tool after one bad output or API timeout; no stickiness | High | High |
| 2 | **Podcast Audience / Listeners** | Indirect — they consume the generated MP3s | Accurate, engaging audio that faithfully represents the source material | Hallucinated facts or clipped audio erode trust in the creator's brand | Low | High |
| 3 | **Original Content Authors / Publishers** | Indirect — their content is ingested and repurposed | Attribution and respect for copyright; no commercial redistribution without licence | Copyright claim or DMCA takedown if the tool is used to monetise scraped content | Low | Medium |
| 4 | **Infrastructure / DevOps Owner** | Indirect — whoever deploys and maintains the app | A stable, observable service with secrets not stored in plain-text `.env` files on a shared machine | API keys leaked; no alerting when the service crashes at 2 am | High | Medium |
| 5 | **Finance / Budget Owner** | Indirect — pays the API bills | Predictable, capped cost per episode (~$0.05); no runaway usage | A bug that loops the pipeline 1000 times racks up hundreds of dollars in LLM charges with no circuit-breaker | High | Low |

---
## Part 3 - From Demo to "Real Project"

### Dimension 1 — Operations (monitoring & incident response)

**Current state:** The app logs every run to `test_audio/run_log.jsonl` and prints tracebacks to stdout, but there is no health-check endpoint, no alerting, and no on-call process. If the Gradio process crashes or an API key expires, the tool is silently broken until someone notices.

**What would change for production:** Add a `/health` endpoint that tests a lightweight API call to each provider on startup and on a 5-minute cron. Pipe error-status log lines to an alerting channel (Slack webhook or PagerDuty). Define a clear runbook: which errors are user-facing (bad URL, empty PDF), which are operational (API outage), and who owns each. The existing `run_log.jsonl` would feed into a dashboard (e.g. Grafana) tracking error rate, latency by provider, and daily episode volume.

---

### Dimension 2 — Security & Compliance (secrets handling, access control)

**Current state:** API keys for Anthropic, OpenAI, and ElevenLabs are stored in a plain-text `.env` file committed to (or adjacent to) the repository. Any team member or CI runner with repo access can read them. There is no authentication on the Gradio UI — anyone with the share URL can run the pipeline and spend API budget.

**What would change for production:** Rotate all keys immediately and move them to a secrets manager (AWS Secrets Manager, HashiCorp Vault, or at minimum GitHub Encrypted Secrets for CI). Add a `GRADIO_USERNAME` / `GRADIO_PASSWORD` auth layer (Gradio supports this natively via `auth=` in `launch()`). For a multi-tenant deployment, scope each API key per tenant and add per-user rate limiting so one user cannot exhaust the shared ElevenLabs character quota.

---

### Dimension 3 — Data Lifecycle (retention, PII, logs)

**Current state:** Every generated script — including the full dialogue text — is stored indefinitely in `run_log.jsonl`. Generated MP3 files accumulate in `test_audio/` with no deletion policy. If a user pastes a document that contains personal data (a CV, a medical summary), that PII lives in the log file forever.

**What would change for production:** Define a retention policy: audio files deleted after 30 days, log entries after 90 days. Strip or hash any user-submitted text before writing it to the log (log metadata — word count, style, duration — not the full source). If the tool is offered as a SaaS product in the EU, a GDPR-compliant deletion workflow is mandatory: a user must be able to request deletion of all their generated content and associated logs within 30 days.

---

### Dimension 4 — Scope beyond the demo (API failures, edge cases, degraded modes)

**Current state:** The demo assumes both the LLM and TTS APIs are reachable. The ElevenLabs fallback to `edge-tts` is implemented for Alex's voice, but if the entire edge-tts library fails (network, Windows firewall, ffmpeg binary missing), the pipeline crashes with an unhandled exception. URL scraping assumes `<p>` tags exist in the page; many JavaScript-rendered sites return an empty body to a simple `requests.get()` call.

**What would change for production:** Add explicit timeout and retry logic (exponential backoff with jitter) for every external API call. For URL scraping, add a secondary extraction strategy using `playwright` for JS-rendered pages, with a hard timeout and a user-facing message when extraction genuinely fails. Add a fully offline fallback mode: if all TTS APIs are unreachable, generate a plain-text transcript and notify the user that audio will be available once connectivity is restored — rather than crashing silently.

---

### Dimension 5 — Handoff (documentation & client training)

**Current state:** The `README.md` covers setup steps, architecture, and the tech stack clearly, but it is written for a developer who will run the app locally. There is no operator guide, no runbook for common failures, no documented process for rotating API keys, and no definition of "done" for a support handoff to a non-technical client.

**What would change for production:** Produce a two-track documentation set: (1) a **user guide** for the content creator (how to get good results from each source type, what the rating system is for, what to do when an episode fails), and (2) an **operator guide** for whoever manages the deployment (how to rotate keys, how to read the run log, how to adjust cost limits). Schedule a 1-hour handoff session with the client before go-live, and define a 30-day support window with agreed SLA (e.g. P1 = pipeline completely down → 4-hour response).

---
## Part 4 — Revision Brief

### Before

The goal of PodcastIQ was to prove that AI can replace the most tedious parts of podcast production. Success meant: a working Gradio UI that accepts four source types and returns a downloadable MP3 in under 60 seconds, with two distinct voices, a humanized script that sounds natural, and a content guard that blocks harmful inputs. If the demo ran end-to-end without crashing during the final presentation, the project was done. Cost, uptime, data handling, and who would actually operate the tool after the bootcamp were out of scope.

### After

Having mapped stakeholders and production gaps, I would reframe success around three concrete additions: (1) a **cost circuit-breaker** — a per-session API spend cap (e.g. $0.20 max) to protect the finance stakeholder and prevent runaway charges from bugs or abuse; (2) a **secrets and access review gate** — no deployment without moving all API keys out of `.env` and behind an auth layer on the Gradio UI, making the DevOps and security stakeholders sign off; and (3) a **data retention policy documented and agreed before launch** — the run log stores full scripts and may inadvertently capture PII, which is a compliance liability rather than a logging convenience. The MVP scope itself would narrow: we would drop the ElevenLabs premium voice in the initial release (to avoid the 401/402 errors already logged) and ship with `edge-tts` only, adding ElevenLabs back in a validated v1.1 once billing and key management are properly handled.

---
## Checklist

- [x] **Recognize:** Named the project (PodcastIQ) and stated one clear intended beneficiary (content creators who want to repurpose written/video content as audio).
- [x] **Apply:** Stakeholder table has 8 rows (≥6), each with Role, Need, Risk, Influence, and Interest.
- [x] **Integrate:** "Demo → real" section covers 5 distinct dimensions — Operations, Security, Data Lifecycle, Scope/Edge Cases, Handoff — each with concrete "what would change" language.
- [x] **Verify:** Revision brief shows a visible shift: from "demo runs without crashing = done" to specific, measurable gates (cost cap, secrets review, data retention policy, scoped MVP).
- [x] **Stretch:** Row 8 — user with accessibility needs — is the easy-to-forget stakeholder. They are the primary beneficiary of audio-first content but are completely unserved by the current UI (no ARIA labels, no captions, no pronunciation control for technical terms).